# Step 1 — **3DAffordSplat / AffordSplat** (local mirror, automatic)

**Pipeline position:** data loading (before reconstruction / rendering notebooks).

This notebook **does not download** anything. It expects the official on-disk tree (Hugging Face checkout), e.g. **`/data/Seen/train/<category>/Gaussian/GS_*.ply`**.

**Automatic root:** `AFFORDANCE_AFFORDSPLAT_ROOT` → `paths.affordsplat_root` → **`AFFORDANCE_DATA_ROOT`** (or config `data_root`) when that folder contains **`Seen/`** → **`/workspace/data`** → **`/data`**. Typical Docker: unzip under **`/workspace/data/Seen`** and re-run.

**Optional env:** `AFFORDANCE_AFFORDSPLAT_SUBSET` (default `Seen`), `AFFORDANCE_AFFORDSPLAT_SPLIT` (`train` / `val` / `test`).

**Random each run:** the dataset cell uses **`shuffle_rows=True`** (new order every kernel run) and picks a **random index** for the printed sample. Re-run that cell to see another object/verb row.

**Pass checklist:** printed root; `len(ds) > 0`; sampled `splat_path` exists on disk.

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
for ROOT in [_cwd, *_cwd.parents]:
    if (ROOT / "pyproject.toml").is_file() and (ROOT / "src").is_dir():
        if str(ROOT / "src") not in sys.path:
            sys.path.insert(0, str(ROOT / "src"))
        break
else:
    raise RuntimeError("Run this notebook from the repository (or notebooks/).")

from utils.config import load_config

_cfg = load_config()
print("project OK; config loaded.")


In [ ]:
import os
import random

from datasets.affordsplat_local_dataset import AffordSplatLocalDataset, resolve_affordsplat_root

_root = resolve_affordsplat_root(_cfg)
print("resolve_affordsplat_root():", _root)

if _root is None:
    raise FileNotFoundError(
        "No AffordSplat mirror found (need a directory containing Seen/). "
        "Typical Docker: unzip into /workspace/data so you have /workspace/data/Seen/... "
        "Or: export AFFORDANCE_DATA_ROOT=/workspace/data "
        "or AFFORDANCE_AFFORDSPLAT_ROOT=/path/parent/of/Seen. "
        "Or set paths.affordsplat_root / paths.data_root in configs/default.yaml."
    )

AFFORDSPLAT_SUBSET = os.environ.get("AFFORDANCE_AFFORDSPLAT_SUBSET", "Seen")
AFFORDSPLAT_SPLIT = os.environ.get("AFFORDANCE_AFFORDSPLAT_SPLIT", "train")
# Restrict categories (optional): e.g. categories=["bag", "bed"]
AFFORDSPLAT_CATEGORIES = None

ds = AffordSplatLocalDataset(
    cfg=_cfg,
    subset=AFFORDSPLAT_SUBSET,
    split=AFFORDSPLAT_SPLIT,
    categories=AFFORDSPLAT_CATEGORIES,
    shuffle_rows=True,
    shuffle_seed=None,
)

print("len(dataset):", len(ds))
print("affordsplat_root:", ds.affordsplat_root)

idx = random.randrange(len(ds))
s0 = ds[idx]
print("random index:", idx)
print("sample[idx] sample_id:", s0["sample_id"])
print("sample[idx] verb:", s0["verb"])
print("sample[idx] split:", s0["split"])
print("sample[idx] splat_path:", s0["splat_path"])
print("splat exists:", s0["splat_path"].is_file())


In [ ]:
# First few rows (verb / category / Gaussian file).
n = min(8, len(ds))
for i in range(n):
    r = ds.row(i)
    print(f"{i:4d}  {r.verb!s:14s}  {r.category!s:12s}  {r.splat_path.name}")


## Optional: `DataLoader`

Samples are **dicts** (paths, strings, `extras`). Use `batch_size=1` or a **custom `collate_fn`** if you batch tensors later.

In [ ]:
from torch.utils.data import DataLoader


def _collate_single_dict(batch: list) -> dict:
    assert len(batch) == 1
    return batch[0]


loader = DataLoader(ds, batch_size=1, shuffle=True, collate_fn=_collate_single_dict)
b = next(iter(loader))
print("keys:", sorted(b.keys()))
print("sample_id:", b["sample_id"])
print("splat_path:", b["splat_path"])
